In [1]:
import pandas as pd
import spacy
# from date_spacy import find_dates
import re

!spacy download en_core_web_sm

In [2]:
# Load the spaCy model for Named Entity Recognition (NER)
nlp = spacy.load("en_core_web_trf")

ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True}, before="ner")

# Add Social Security Number
ssn_pattern_regex = {
    "label": "SSN",
    "pattern": [
        {"TEXT": {"REGEX": "\\d{3}"}},
        {"TEXT": "-"},
        {"TEXT": {"REGEX": "\\d{2}"}},
        {"TEXT": "-"},
        {"TEXT": {"REGEX": "\\d{4}"}}
    ]
}
# Add gender/sex pattern
gender_pattern_regex = {
    "label": "GENDER",
    "pattern": [
        {"LOWER": {"REGEX": "\\b(gender|sex|male|female|man|woman|boy|girl|he|she|him|her)\\b"}}
    ]
}
ruler.add_patterns([ssn_pattern_regex, gender_pattern_regex])


In [3]:
def regex_name_fallback(text, redacted_text):
    # Match words that look like names (e.g., lowercase words in a sentence)
    name_like_words = re.findall(r'\b[a-z][a-z]+\b', text)
    for word in name_like_words:
        if word in text and word not in redacted_text:
            redacted_text = re.sub(rf'\b{word}\b', '[]', redacted_text)
    return redacted_text

In [4]:
# Sample dataframe
data = {'comments': [
    "My credit card number is 1234-5678-9012-3456",
    "My date of birth is 01/01/1990",
    "My driver's license number is A1234567",
    "My financial information includes bank account number 123456789",
    "The full name is John Doe",
    "My gender is male",
    "My mailing address is 123 Main St, Anytown, USA",
    "My medical records show I have diabetes",
    "My passport information is passport number 987654321",
    "My place of birth is Anytown, USA",
    "My race is Caucasian",
    "My religion is Christianity",
    "My Social Security number (SSN) is 123-45-6789",
    "My ZIP code is 12345"
]}


In [5]:
df = pd.DataFrame(data)
df

,comments
0,My credit card number is 1234-5678-9012-3456
1,My date of birth is 01/01/1990
2,My driver's license number is A1234567
3,My financial information includes bank account...
4,The full name is John Doe
5,My gender is male
6,"My mailing address is 123 Main St, Anytown, USA"
7,My medical records show I have diabetes
8,My passport information is passport number 987...
9,"My place of birth is Anytown, USA"


In [ ]:
# Define a function to redact PII using spaCy NER
def redact_pii(text):
    doc = nlp(text)
    redacted_text = text
    for ent in doc.ents:
        if ent.label_ in ["SSN", 
                          "GENDER", 
                          "PERSON", 
                          "NORP", 
                        #   "FAC", 
                          "ORG", 
                          "GPE", 
                          # "LOC", 
                          # "PRODUCT", 
                          "EVENT", 
                          # "WORK_OF_ART", 
                          "LAW", 
                          "LANGUAGE", 
                          "DATE", 
                          "TIME", 
                          # "PERCENT", 
                          # "MONEY", 
                          # "QUANTITY", 
                          # "ORDINAL", 
                          # "CARDINAL"
                          ]:
            redacted_text = redacted_text.replace(ent.text, "[REDACTED]")
        else:
            redacted_text = redacted_text.replace(ent.text, "[REDACTED]")
    return redacted_text


In [7]:
# Apply the function to the comments column
for comment in df["comments"].to_list():
    print(comment)
    print(redact_pii(comment))


My credit card number is 1234-5678-9012-3456
My credit card number is [REDACTED]-3456
My date of birth is 01/01/1990
My date of birth is [REDACTED]
My driver's license number is A1234567
My driver's license number is A1234567
My financial information includes bank account number 123456789
My financial information includes bank account number [REDACTED]
The full name is John Doe
The full name is [REDACTED]
My gender is male
My [REDACTED] is [REDACTED]
My mailing address is 123 Main St, Anytown, USA
My mailing address is 123 Main St, [REDACTED], [REDACTED]
My medical records show I have diabetes
My medical records show I have diabetes
My passport information is passport number 987654321
My passport information is passport number [REDACTED]
My place of birth is Anytown, USA
My place of birth is [REDACTED]
My race is Caucasian
My race is [REDACTED]
My religion is Christianity
My religion is Christianity
My Social Security number (SSN) is 123-45-6789
My [REDACTED] number (SSN) is [REDACTED]

In [8]:
doc = nlp("There's no overall DT project progress shared with regions with andrew, regions are asking more but no regular official information provided.")
for ent in doc.ents:
    print(ent.text, ent.label_)

regions ORG
andrew ORG
